In [ ]:
!pip install transformers torch scikit-learn -qq

In [ ]:
# Install necessary libraries: transformers for NLP models, torch for deep learning, and scikit-learn for metrics.
!pip install transformers torch scikit-learn -qq

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import drive
drive.mount('/content/drive') # Mount Google Drive to access stored datasets and models.

In [ ]:
# Imports
import os
import random
import time
import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import AutoTokenizer, BertForSequenceClassification
from transformers.optimization import get_linear_schedule_with_warmup

from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, recall_score, precision_score
)

# Reproducibility
seed_val = 42
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# Imports
import os
import random
import time
import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import AutoTokenizer, BertForSequenceClassification
from transformers.optimization import get_linear_schedule_with_warmup

from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, recall_score, precision_score
)

# Reproducibility: Set seeds for random number generators to ensure consistent results.
seed_val = 42
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)

# Device: Determine if a GPU (cuda) is available and use it; otherwise, fall back to CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Grabbed from other colab notebook with the baseline models
train = pd.read_csv('/content/drive/MyDrive/train.csv')
val   = pd.read_csv('/content/drive/MyDrive/val.csv')
test  = pd.read_csv('/content/drive/MyDrive/test.csv')

# df used only for visualizations later
df = pd.concat([train, val, test]).reset_index(drop=True)

print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")
print(f"Total: {len(df):,}")
print(df['label'].value_counts())

Train: 613,788 | Val: 131,526 | Test: 131,526
Total: 876,840
label
1.0    471180
0.0    405660
Name: count, dtype: int64


In [ ]:
# Grabbed from the baseline notebooks' partitioning.
train = pd.read_csv('/content/drive/MyDrive/train.csv')
val   = pd.read_csv('/content/drive/MyDrive/val.csv')
test  = pd.read_csv('/content/drive/MyDrive/test.csv')

# Combine all datasets into a single DataFrame
df = pd.concat([train, val, test]).reset_index(drop=True)

print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")
print(f"Total: {len(df):,}")
print(df['label'].value_counts())

In [ ]:
# Prepare the data: Convert labels to integer type for consistency.
for split in [train, val, test]:
    split['label'] = split['label'].astype(int)

# Load the pre-trained 'bert-base-uncased' tokenizer.
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Define a function to tokenize text data.
def tokenize_function(texts):
    # Pad to max_length, truncate longer sequences, and return PyTorch tensors.
    return tokenizer(list(texts), padding='max_length', truncation=True,
                     max_length=128, return_tensors='pt')

print("Tokenizing train...")
train_enc = tokenize_function(train['text'])
print("Tokenizing val...")
val_enc   = tokenize_function(val['text'])
print("Tokenizing test...")
test_enc  = tokenize_function(test['text'])

# Extract input IDs and attention masks from the tokenized outputs.
train_inputs, train_masks = train_enc['input_ids'], train_enc['attention_mask']
val_inputs,   val_masks   = val_enc['input_ids'],   val_enc['attention_mask']
test_inputs,  test_masks  = test_enc['input_ids'],  test_enc['attention_mask']

# Convert labels to PyTorch tensors.
train_labels = torch.tensor(train['label'].values)
val_labels   = torch.tensor(val['label'].values)
test_labels  = torch.tensor(test['label'].values)

# Print the shapes of the tokenized inputs and the label distribution for the training set.
print(f"Train: {train_inputs.shape} | Val: {val_inputs.shape} | Test: {test_inputs.shape}")
print(f"Label distribution: {np.bincount(train_labels.numpy())}")

In [ ]:
class RedditDepressionDataset(Dataset):
    def __init__(self, input_ids, attention_masks, labels):
        self.input_ids = input_ids
        self.attention_masks = attention_masks
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_masks[idx],
            'labels': self.labels[idx]
        }

batch_size = 32

train_dataset = RedditDepressionDataset(train_inputs, train_masks, train_labels)
val_dataset   = RedditDepressionDataset(val_inputs,   val_masks,   val_labels)
test_dataset  = RedditDepressionDataset(test_inputs,  test_masks,  test_labels)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_dataloader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

loss_fn = torch.nn.CrossEntropyLoss()

print(f"Train batches: {len(train_dataloader)}")
print(f"Val batches:   {len(val_dataloader)}")
print(f"Test batches:  {len(test_dataloader)}")

Train batches: 19181
Val batches:   4111
Test batches:  4111


In [ ]:
class RedditDepressionDataset(Dataset):
    # Custom PyTorch Dataset class to handle the input data.
    def __init__(self, input_ids, attention_masks, labels):
        self.input_ids = input_ids
        self.attention_masks = attention_masks
        self.labels = labels

    def __len__(self):
        # Returns the total number of samples in the dataset.
        return len(self.labels)

    def __getitem__(self, idx):
        # Returns a sample at the given index.
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_masks[idx],
            'labels': self.labels[idx]
        }

batch_size = 32

# Create instances of the custom dataset for each split.
train_dataset = RedditDepressionDataset(train_inputs, train_masks, train_labels)
val_dataset   = RedditDepressionDataset(val_inputs,   val_masks,   val_labels)
test_dataset  = RedditDepressionDataset(test_inputs,  test_masks,  test_labels)

# Create DataLoaders to efficiently load data in batches during training/evaluation.
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_dataloader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

# loss function for classification
loss_fn = torch.nn.CrossEntropyLoss()

# Print the number of batches for each dataloader.
print(f"Train batches: {len(train_dataloader)}")
print(f"Val batches:   {len(val_dataloader)}")
print(f"Test batches:  {len(test_dataloader)}")

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2,
    output_attentions=False,
    output_hidden_states=False,
)
model.to(device)
print(f"Model loaded on {device}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded on cuda


In [ ]:
# Load the pre-trained BERT model for sequence classification.
# 'bert-base-uncased' is the base model, num_labels=2 for binary classification.
# output_attentions and output_hidden_states are set to False as they are not needed for this task.
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2,
    output_attentions=False,
    output_hidden_states=False,
)
model.to(device) # Move the model to GPU
print(f"Model loaded on {device}")

In [ ]:
epochs = 2

# Initialize the AdamW optimizer with specified learning rate and epsilon.
optimizer = AdamW(model.parameters(), lr=1e-5, eps=1e-8)

# Calculate the total number of training steps.
total_steps = len(train_dataloader) * epochs

# Set up the learning rate scheduler.
# linearly increases LR then decreases it.
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

# Helper function to calculate accuracy.
def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

# Helper function to format time elapsed.
def format_time(elapsed):
    return str(datetime.timedelta(seconds=int(round(elapsed))))

print(f"Total training steps: {total_steps}")

In [ ]:
history = {
    'train_loss': [],
    'val_loss': [],
    'val_accuracy': []
}

best_val_loss = float('inf') # Initialize with a very high value for tracking the best validation loss.
patience = 2 # Number of epochs to wait for improvement before early stopping.
epochs_no_improve = 0 # Counter for epochs without improvement.

print('Starting training...')

# Main training loop.
for epoch_i in range(epochs):
    print(f'\n======== Epoch {epoch_i + 1} / {epochs} ========')
    print('Training...')

    t0 = time.time() # Record time for epoch.
    total_train_loss = 0
    model.train() # Set the model to training mode.

    # Iterate over batches in the training dataloader.
    for step, batch in enumerate(train_dataloader):
        # Print progress every 500 batches.
        if step % 500 == 0 and step != 0:
            elapsed = format_time(time.time() - t0)
            print(f'  Batch {step:>5,} of {len(train_dataloader):>5,}. Elapsed: {elapsed}.')

        # Move batch data to the correct device.
        b_input_ids  = batch['input_ids'].to(device)
        b_input_mask = batch['attention_mask'].to(device)
        b_labels     = batch['labels'].to(device)

        model.zero_grad() # Clear any previously calculated gradients.
        # Perform a forward pass and get model outputs
        outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask)
        loss = loss_fn(outputs.logits, b_labels)
        total_train_loss += loss.item()
        loss.backward() # Perform backward pass to compute gradients.
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Clip gradients to prevent exploding gradients.
        optimizer.step()
        scheduler.step()

    avg_train_loss = total_train_loss / len(train_dataloader) # Calculate average training loss.
    print(f'  Average training loss: {avg_train_loss:.2f}')
    print(f'  Training epoch took: {format_time(time.time() - t0)}')

    print('\nValidating...')
    t0 = time.time() # Record time for validation.
    model.eval() # Set the model to evaluation mode.
    total_eval_accuracy = 0
    total_eval_loss = 0

    # Iterate over batches in the validation dataloader.
    for batch in val_dataloader:
        # Move batch data to the correct device.
        b_input_ids  = batch['input_ids'].to(device)
        b_input_mask = batch['attention_mask'].to(device)
        b_labels     = batch['labels'].to(device)

        # Disable gradient calculations during validation.
        with torch.no_grad():
            outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask)

        loss = loss_fn(outputs.logits, b_labels)
        total_eval_loss += loss.item()
        logits = outputs.logits.detach().cpu().numpy() # Move logits to CPU and convert to numpy array.
        label_ids = b_labels.to('cpu').numpy() # Move labels to CPU and convert to numpy array.
        total_eval_accuracy += flat_accuracy(logits, label_ids)

    avg_val_accuracy = total_eval_accuracy / len(val_dataloader)
    avg_val_loss = total_eval_loss / len(val_dataloader)
    print(f'  Accuracy: {avg_val_accuracy:.4f}')
    print(f'  Validation Loss: {avg_val_loss:.4f}')
    print(f'  Validation took: {format_time(time.time() - t0)}')

    # Store epoch statistics in history.
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['val_accuracy'].append(avg_val_accuracy)

    # Check for improvement in validation loss for early stopping.
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_model.pt') # Save the model state dictionary if it's the best so far.
        print('  Saved best model.')
        epochs_no_improve = 0 # Reset patience counter.
    else:
        epochs_no_improve += 1
        print(f'  No improvement for {epochs_no_improve} epoch(s).')
        if epochs_no_improve >= patience:
            print('Early stopping triggered!')
            break # Stop training if patience runs out.

print('\nTraining complete!')

In [ ]:
model.load_state_dict(torch.load('best_model.pt')) # Load the weights from the best performing model saved during training.
print('Loaded best model weights.')

model.save_pretrained('/content/drive/MyDrive/bert_depression_model') # Save the entire model (architecture + weights).
tokenizer.save_pretrained('/content/drive/MyDrive/bert_depression_model') # Save the tokenizer used with the model.
print('Saved to Drive.')

In [ ]:
print('Evaluating on Test Set...')
model.eval()

all_preds = []
all_labels = []

for batch in test_dataloader:
    b_input_ids  = batch['input_ids'].to(device)
    b_input_mask = batch['attention_mask'].to(device)
    b_labels     = batch['labels'].to(device)

    with torch.no_grad():
        outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask)

    logits = outputs.logits.detach().cpu().numpy()
    label_ids = b_labels.to('cpu').numpy()
    all_preds.extend(np.argmax(logits, axis=1).flatten())
    all_labels.extend(label_ids.flatten())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

print(f'  Accuracy:  {np.sum(all_preds == all_labels) / len(all_labels):.4f}')
print(f'  Precision: {precision_score(all_labels, all_preds):.4f}')
print(f'  Recall:    {recall_score(all_labels, all_preds):.4f}')
print(f'  F1:        {f1_score(all_labels, all_preds):.4f}')
print('\nClassification Report:')
print(classification_report(all_labels, all_preds, target_names=['Non-depressed', 'Depressed']))

Evaluating on Test Set...
  Accuracy:  0.9435
  Precision: 0.9407
  Recall:    0.9550
  F1:        0.9478

Classification Report:
               precision    recall  f1-score   support

Non-depressed       0.95      0.93      0.94     60849
    Depressed       0.94      0.96      0.95     70677

     accuracy                           0.94    131526
    macro avg       0.94      0.94      0.94    131526
 weighted avg       0.94      0.94      0.94    131526



In [1]:
# Identify misclassified posts
print('\nIdentifying misclassified posts...')
misclassified_indices = np.where(all_preds != all_labels)[0]

misclassified_df = test.iloc[misclassified_indices].copy()
misclassified_df['predicted_label'] = all_preds[misclassified_indices]

print(f'Total misclassified posts: {len(misclassified_df):,}')

# Display a few examples of misclassified posts
print('\nExamples of misclassified posts:')
display(misclassified_df[['text', 'label', 'predicted_label']].head(10))

# Separate misclassified by label
misclassified_as_depressed_but_not = misclassified_df[(misclassified_df['label'] == 0) & (misclassified_df['predicted_label'] == 1)]
misclassified_as_not_depressed_but_is = misclassified_df[(misclassified_df['label'] == 1) & (misclassified_df['predicted_label'] == 0)]

print(f'\nPosts incorrectly classified as Depressed (True: Non-depressed): {len(misclassified_as_depressed_but_not):,}')
display(misclassified_as_depressed_but_not[['text', 'label', 'predicted_label']].head(5))

print(f'\nPosts incorrectly classified as Non-depressed (True: Depressed): {len(misclassified_as_not_depressed_but_is):,}')
display(misclassified_as_not_depressed_but_is[['text', 'label', 'predicted_label']].head(5))


Identifying misclassified posts...


NameError: name 'np' is not defined